# Pilot Model v3 — две candidate configurations

Обе конфигурации — **pilot candidates**, accepted feature registry не меняется. Sep–Oct validation служит только диагностике: её WAPE не управляет final training и не выбирает submission. После диагностики обе модели независимо обучаются на всей известной Jan–Oct истории и строят отдельные Nov–Dec submissions. Единица наблюдения — `(route, date, hour)`, target — `boardings`. Используются готовые model-ready CSV; raw labels, grid и features заново не строятся. Observed weather 2025 не используется.

## Environment & Data Setup

Напрямую загружаем два обновлённых model-ready CSV согласно `data/processed/README_model_ready.md`: UTF-8, разделитель `;`, `date` как datetime. Категории `temperature_bucket` восстанавливаются при чтении в том же порядке и с тем же ordered categorical dtype, что и в `pilot_model_v3_experiment.py`. Путь определяется от текущего каталога notebook или корня репозитория; в Colab поддержан путь v1 через Google Drive. Версии библиотек выводятся для воспроизводимости.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm
from IPython.display import display
from lightgbm import LGBMRegressor

def find_data_root():
    candidates = [parent / 'data' for parent in (Path.cwd(), *Path.cwd().parents)]
    candidates.append(Path('/content/drive/MyDrive/MT-Hackathon-2026/data'))
    for candidate in candidates:
        if (candidate / 'processed' / 'model_ready_train_2025_jan_oct.csv').is_file():
            return candidate
    raise FileNotFoundError('Model-ready data directory not found')

if Path('/content').exists() and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

DATA_ROOT = find_data_root()
PROCESSED_DIR = DATA_ROOT / 'processed'
TRAIN_PATH = PROCESSED_DIR / 'model_ready_train_2025_jan_oct.csv'
FORECAST_PATH = PROCESSED_DIR / 'model_ready_forecast_2025_nov_dec.csv'
SUBMISSION_PATHS = {
    'calendar_weather_candidate': PROCESSED_DIR / 'pilot_model_v3_calendar_weather_submission.csv',
    'weather_only_candidate': PROCESSED_DIR / 'pilot_model_v3_weather_only_submission.csv',
}

BUCKET_DTYPE = pd.CategoricalDtype(categories=('<0', '[0,10)', '[10,20)', '>=20'), ordered=True)
train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8', parse_dates=['date'], dtype={'temperature_bucket': BUCKET_DTYPE})
forecast = pd.read_csv(FORECAST_PATH, sep=';', encoding='utf-8', parse_dates=['date'], dtype={'temperature_bucket': BUCKET_DTYPE})
print('pandas:', pd.__version__, 'LightGBM:', lightgbm.__version__)
print('train rows:', len(train), 'forecast rows:', len(forecast))

pandas: 2.2.3 LightGBM: 4.6.0
train rows: 72960 forecast rows: 14640


## Data Contract / Sanity Checks

Проверки останавливают pipeline при изменении входного контракта: нужны ровно указанные столбцы, все дни периода, 10 маршрутов × 24 часа, уникальные ключи и отсутствие пропусков. Forecast не должен содержать target. `temperature_bucket` сверяется с frozen `climatological_temperature` и заданными границами. Это защищает сравнение и оба submission от сдвига схемы или сетки.

In [11]:
ROUTES = (1, 5, 7, 11, 12, 17, 25, 26, 28, 50)
KEYS = ['route', 'date', 'hour']
SET_FEATURES = (
    'route', 'weekday', 'hour', 'route_hour', 'hour_weekend', 'is_night'
)
CALENDAR_FEATURES = (
    'is_day_off', 'is_official_holiday', 'is_transferred_day_off',
    'is_transferred_workday', 'is_shortened_workday'
)
WEATHER_FEATURES = ('temperature_bucket',)
CALENDAR_WEATHER_FEATURES = SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES
WEATHER_ONLY_FEATURES = SET_FEATURES + WEATHER_FEATURES
CANDIDATE_CONFIGURATIONS = {
    'calendar_weather_candidate': CALENDAR_WEATHER_FEATURES,
    'weather_only_candidate': WEATHER_ONLY_FEATURES,
}
WEATHER_COLUMNS = (
    'climatological_temperature', 'climatological_precipitation',
    'climatological_rain_probability', 'temperature_bucket'
)
BUCKET_LABELS = ('<0', '[0,10)', '[10,20)', '>=20')
FORECAST_COLUMNS = [
    'route', 'date', 'hour', 'weekday', 'route_hour', 'hour_weekend',
    'is_night', *CALENDAR_FEATURES, *WEATHER_COLUMNS
]
TRAIN_COLUMNS = FORECAST_COLUMNS[:-len(WEATHER_COLUMNS)] + ['boardings', *WEATHER_COLUMNS]

if train.columns.tolist() != TRAIN_COLUMNS:
    raise ValueError('Train schema differs from model-ready contract')
if forecast.columns.tolist() != FORECAST_COLUMNS:
    raise ValueError('Forecast schema differs from model-ready contract')
if train.drop(columns='boardings').columns.tolist() != forecast.columns.tolist():
    raise ValueError('Train and forecast feature columns differ')
if set(train.columns) - set(KEYS) - {'boardings'} != set(SET_FEATURES + CALENDAR_FEATURES + WEATHER_COLUMNS) - set(KEYS):
    raise ValueError('Unexpected feature columns')

def check_grid(frame, start, end, expected_rows):
    dates = pd.date_range(start, end, freq='D')
    if len(frame) != expected_rows or frame[KEYS].isna().any().any():
        raise ValueError('Incorrect row count or missing keys')
    if frame.isna().any().any():
        raise ValueError('Missing values in model-ready data')
    if not pd.api.types.is_datetime64_any_dtype(frame['date']):
        raise TypeError('date must be datetime')
    if frame[KEYS].duplicated().any():
        raise ValueError('Duplicate route/date/hour keys')
    if set(frame['route']) != set(ROUTES) or set(frame['hour']) != set(range(24)):
        raise ValueError('Unexpected route or hour values')
    if set(frame['date']) != set(dates):
        raise ValueError('Incomplete or unexpected date period')
    if len(frame) != len(ROUTES) * len(dates) * 24:
        raise ValueError('Incomplete route/date/hour grid')
    if not all(pd.api.types.is_numeric_dtype(frame[col]) for col in frame.columns if col not in ('date', 'temperature_bucket')):
        raise TypeError('Non-numeric feature or target')
    if frame['temperature_bucket'].dtype != BUCKET_DTYPE:
        raise TypeError('Temperature bucket categorical dtype changed')
    if not frame['temperature_bucket'].isin(BUCKET_LABELS).all():
        raise ValueError('Unknown temperature bucket')
    expected_bucket = pd.cut(
        frame['climatological_temperature'], bins=[-np.inf, 0, 10, 20, np.inf],
        right=False, labels=BUCKET_LABELS,
    )
    if not frame['temperature_bucket'].eq(expected_bucket).all():
        raise ValueError('Temperature bucket differs from frozen climatology')
    if not frame['weekday'].eq(frame['date'].dt.dayofweek).all():
        raise ValueError('weekday does not match date')
    if not frame['route_hour'].eq(frame['route'] * 24 + frame['hour']).all():
        raise ValueError('route_hour encoding differs from v1')
    if not frame['hour_weekend'].eq(frame['hour'] * 2 + frame['weekday'].ge(5).astype('int8')).all():
        raise ValueError('hour_weekend encoding differs from v1')
    if not frame['is_night'].eq(frame['hour'].between(0, 5).astype('int8')).all():
        raise ValueError('is_night encoding differs from v1')
    for col in ('is_night', *CALENDAR_FEATURES):
        if not frame[col].isin((0, 1)).all():
            raise ValueError(f'{col} must be binary')

check_grid(train, '2025-01-01', '2025-10-31', 72_960)
check_grid(forecast, '2025-11-01', '2025-12-31', 14_640)
if 'boardings' in forecast.columns:
    raise ValueError('Forecast must not contain target')
if train['boardings'].lt(0).any():
    raise ValueError('Negative training target')
print('Model-ready data contract passed.')

Model-ready data contract passed.


## Feature Configuration

`SET_FEATURES` — frozen accepted core. Две независимые candidate configurations: `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES` и `SET_FEATURES + WEATHER_FEATURES`. Только `temperature_bucket` из weather columns входит в модель; он остаётся ordered categorical с метками `<0`, `[0,10)`, `[10,20)`, `>=20`, как в проверенном v3 experiment. Пять core категориальных полей сохраняют фиксированные категории v2; `is_night` и calendar flags остаются числовыми. Observed weather 2025 и historical H19–H27 признаки не используются.

In [12]:
CATEGORICAL_FEATURES = ('route', 'weekday', 'hour', 'route_hour', 'hour_weekend')
CATEGORIES = {
    'route': ROUTES,
    'weekday': tuple(range(7)),
    'hour': tuple(range(24)),
    'route_hour': tuple(route * 24 + hour for route in ROUTES for hour in range(24)),
    'hour_weekend': tuple(range(48)),
}
MODEL_PARAMS = {
    'objective': 'regression_l1',
    'n_estimators': 300,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.1,
    'random_state': 42,
    'n_jobs': 4,
    'verbosity': -1,
}

def make_features(rows, feature_set):
    features = rows.loc[:, list(feature_set)].copy()
    for col in CATEGORICAL_FEATURES:
        features[col] = pd.Categorical(features[col], categories=CATEGORIES[col])
        if features[col].isna().any():
            raise ValueError(f'Unknown category in {col}')
    if 'temperature_bucket' in feature_set:
        features['temperature_bucket'] = features['temperature_bucket'].astype(BUCKET_DTYPE)
        if features['temperature_bucket'].isna().any():
            raise ValueError('Unknown temperature bucket')
    for col in set(feature_set) - set(CATEGORICAL_FEATURES) - {'temperature_bucket'}:
        features[col] = features[col].astype('int8')
    return features

for feature_set in (SET_FEATURES, *CANDIDATE_CONFIGURATIONS.values()):
    for frame in (train, forecast):
        prepared = make_features(frame, feature_set)
        if prepared.columns.tolist() != list(feature_set):
            raise ValueError('Feature order mismatch')
        if any(not isinstance(prepared[col].dtype, pd.CategoricalDtype) for col in CATEGORICAL_FEATURES):
            raise TypeError('Categorical preprocessing mismatch')
        if 'temperature_bucket' in feature_set and prepared['temperature_bucket'].dtype != BUCKET_DTYPE:
            raise TypeError('Temperature bucket preprocessing mismatch')
print('Feature sets and shared preprocessing passed.')

Feature sets and shared preprocessing passed.


## Validation Strategy

Jan–Aug → Sep–Oct — **только диагностический** temporal split. Во всех трёх конфигурациях один split, frozen LightGBM, preprocessing и postprocessing. Target Sep–Oct не попадает в признаки или обучение и используется лишь после predictions для метрик. WAPE = сумма абсолютных ошибок / сумма фактических посадок, WAPE-score = `max(0, 1 − WAPE)`. Для route 5 с нулевой суммой target per-route WAPE не определён (`NaN`). Результат validation не используется ни для выбора финальной модели, ни для условия записи submission.

In [13]:
validation_train = train.loc[train['date'] <= '2025-08-31'].copy()
validation_holdout = train.loc[train['date'] >= '2025-09-01'].copy()
if len(validation_train) != 58_320 or len(validation_holdout) != 14_640:
    raise ValueError('Temporal split has unexpected size')
if validation_train['date'].max() >= validation_holdout['date'].min():
    raise ValueError('Temporal split overlaps')

def make_model():
    return LGBMRegressor(**MODEL_PARAMS)

def postprocess_predictions(raw_predictions, routes):
    prediction = np.floor(np.maximum(np.asarray(raw_predictions), 0) + 0.5).astype('int64')
    prediction[np.asarray(routes) == 5] = 0
    return prediction

def calculate_metrics(actual, prediction, routes):
    actual = np.asarray(actual)
    prediction = np.asarray(prediction)
    total_actual = actual.sum()
    if total_actual <= 0:
        raise ValueError('Overall WAPE is undefined for zero total actual')
    absolute_error = np.abs(actual - prediction)
    wape = absolute_error.sum() / total_actual
    route_frame = pd.DataFrame({
        'route': np.asarray(routes), 'actual': actual, 'absolute_error': absolute_error
    })
    route_metrics = route_frame.groupby('route').agg(
        actual=('actual', 'sum'), absolute_error=('absolute_error', 'sum')
    ).reindex(ROUTES)
    route_metrics['WAPE'] = route_metrics['absolute_error'] / route_metrics['actual'].replace(0, np.nan)
    return wape, max(0.0, 1.0 - wape), route_metrics.reset_index()

## Jan–Aug → Sep–Oct diagnostic validation

Независимо обучаем core reference и две candidate configurations с теми же frozen параметрами. После одинакового округления прогноз route 5 устанавливается в ноль. Показываем overall WAPE, WAPE-score и per-route WAPE без выбора победителя.

In [14]:
validation_results = []
route_results = {}
diagnostic_configurations = {'core_reference': SET_FEATURES, **CANDIDATE_CONFIGURATIONS}
for name, feature_set in diagnostic_configurations.items():
    model = make_model()
    model.fit(
        make_features(validation_train, feature_set), validation_train['boardings'],
        categorical_feature=[*CATEGORICAL_FEATURES, *WEATHER_FEATURES] if 'temperature_bucket' in feature_set else list(CATEGORICAL_FEATURES),
    )
    prediction = postprocess_predictions(
        model.predict(make_features(validation_holdout, feature_set)),
        validation_holdout['route'],
    )
    wape, score, per_route = calculate_metrics(
        validation_holdout['boardings'], prediction, validation_holdout['route']
    )
    validation_results.append({'configuration': name, 'WAPE': wape, 'WAPE-score': score})
    route_results[name] = per_route.set_index('route')['WAPE']

validation_summary = pd.DataFrame(validation_results)
route_wape = pd.DataFrame(route_results)
display(validation_summary.round(6))
display(route_wape.round(6))
print('Diagnostic only: accepted feature registry unchanged; final fits do not depend on these metrics.')

,configuration,WAPE,WAPE-score
0,core_reference,0.123580,0.876420
1,calendar_weather_candidate,0.115756,0.884244
2,weather_only_candidate,0.113909,0.886091


,core_reference,calendar_weather_candidate,weather_only_candidate
route,,,
1,0.109409,0.084371,0.083873
5,NaN,NaN,NaN
7,0.157333,0.163654,0.168162
11,0.105455,0.108475,0.106313
12,0.088066,0.096495,0.093095
17,0.085894,0.073963,0.069796
25,0.199599,0.148178,0.146432
26,0.125388,0.096106,0.095027
28,0.153739,0.131471,0.131434


Diagnostic only: accepted feature registry unchanged; final fits do not depend on these metrics.


## Final Training Jan–Oct

Независимо от diagnostic validation обучаем **обе** candidate configurations на всех 72 960 model-ready строках Jan–Oct. Frozen LightGBM и preprocessing одинаковы; Sep–Oct target здесь входит в полную известную историю. Никакой выбор конфигурации по validation metrics или изменение accepted feature registry не выполняется.

In [15]:
final_models = {}
for name, feature_set in CANDIDATE_CONFIGURATIONS.items():
    model = make_model()
    model.fit(
        make_features(train, feature_set), train['boardings'],
        categorical_feature=[*CATEGORICAL_FEATURES, *WEATHER_FEATURES],
    )
    final_models[name] = model
    print('Final candidate trained:', name, 'rows:', len(train))

Final candidate trained: calendar_weather_candidate rows: 72960
Final candidate trained: weather_only_candidate rows: 72960


## Nov–Dec Inference

Берём **непосредственно** `model_ready_forecast_2025_nov_dec.csv` в исходном порядке. Для каждой независимо обученной модели применяем те же категории, frozen округление `floor(max(raw, 0) + 0.5)` и принудительное `prediction = 0` для route 5. Обе конфигурации прогнозируют весь горизонт из 14 640 строк.

In [16]:
if forecast.columns.tolist() != FORECAST_COLUMNS or 'boardings' in forecast:
    raise ValueError('Forecast feature schema or target presence changed')
inferences = {}
for name, feature_set in CANDIDATE_CONFIGURATIONS.items():
    raw_prediction = final_models[name].predict(make_features(forecast, feature_set))
    if not np.isfinite(raw_prediction).all():
        raise ValueError(f'Non-finite raw predictions: {name}')
    inference = forecast.loc[:, KEYS].copy()
    inference['prediction'] = postprocess_predictions(raw_prediction, forecast['route'])
    if len(inference) != 14_640 or inference.isna().any().any():
        raise ValueError(f'Incomplete inference: {name}')
    if not np.isfinite(inference['prediction']).all() or inference['prediction'].lt(0).any():
        raise ValueError(f'Invalid predictions: {name}')
    if not pd.api.types.is_integer_dtype(inference['prediction']):
        raise TypeError(f'Predictions are not integers: {name}')
    if not inference.loc[inference['route'].eq(5), 'prediction'].eq(0).all():
        raise ValueError(f'Route 5 policy failed: {name}')
    inferences[name] = inference
    print('Predictions generated:', name, len(inference))

Predictions generated: calendar_weather_candidate 14640
Predictions generated: weather_only_candidate 14640


## Submission

Сохраняем два отдельных pilot candidate CSV: `pilot_model_v3_calendar_weather_submission.csv` и `pilot_model_v3_weather_only_submission.csv`. Каждый содержит ровно `route;date;hour;prediction`, UTF-8 и `;`, наследует ключи и порядок forecast без сортировки. В notebook не выбирается победитель.

In [17]:
for name, inference in inferences.items():
    submission = inference.loc[:, ['route', 'date', 'hour', 'prediction']].copy()
    submission['date'] = submission['date'].dt.strftime('%Y-%m-%d')
    submission.to_csv(SUBMISSION_PATHS[name], sep=';', index=False, encoding='utf-8')
    print('Submission saved to:', SUBMISSION_PATHS[name])

Submission saved to: /content/drive/MyDrive/MT-Hackathon-2026/data/processed/pilot_model_v3_calendar_weather_submission.csv
Submission saved to: /content/drive/MyDrive/MT-Hackathon-2026/data/processed/pilot_model_v3_weather_only_submission.csv


## Final Sanity Checks

Повторно читаем **оба** submission artifacts и проверяем формат, число строк, полный Nov–Dec период, маршруты и часы, уникальность ключей, точный порядок forecast, finite целочисленные неотрицательные predictions без NaN и нули route 5. Forecast не содержит target; model feature sets используют только frozen `temperature_bucket`, не observed weather 2025.

In [18]:
expected_keys = forecast[KEYS].copy()
expected_keys['date'] = expected_keys['date'].dt.strftime('%Y-%m-%d')
expected_keys = expected_keys.reset_index(drop=True)
template = pd.read_csv(DATA_ROOT / 'raw' / 'test_submission.csv', sep=';', encoding='utf-8')
if not template[KEYS].equals(expected_keys):
    raise ValueError('Forecast keys/order differ from submission template')
for name, path in SUBMISSION_PATHS.items():
    saved = pd.read_csv(path, sep=';', encoding='utf-8')
    if saved.columns.tolist() != ['route', 'date', 'hour', 'prediction']:
        raise ValueError(f'Saved submission columns or order differ: {name}')
    if len(saved) != 14_640 or saved[KEYS].duplicated().any():
        raise ValueError(f'Saved submission row count or keys are invalid: {name}')
    if saved.isna().any().any():
        raise ValueError(f'Saved submission contains NaN: {name}')
    if set(saved['route']) != set(ROUTES) or set(saved['hour']) != set(range(24)):
        raise ValueError(f'Saved submission routes or hours differ: {name}')
    if set(saved['date']) != set(pd.date_range('2025-11-01', '2025-12-31').strftime('%Y-%m-%d')):
        raise ValueError(f'Saved submission period differs: {name}')
    if not np.isfinite(saved['prediction']).all():
        raise ValueError(f'Saved predictions are non-finite: {name}')
    if saved['prediction'].lt(0).any() or not pd.api.types.is_integer_dtype(saved['prediction']):
        raise ValueError(f'Saved predictions must be non-negative integers: {name}')
    if not saved.loc[saved['route'].eq(5), 'prediction'].eq(0).all():
        raise ValueError(f'Saved route 5 predictions must be zero: {name}')
    if not saved[KEYS].equals(expected_keys):
        raise ValueError(f'Saved keys or row order differ from forecast input: {name}')
    print('Final sanity checks passed:', name, '14,640 Nov-Dec rows')
if 'boardings' in forecast or set(WEATHER_FEATURES) != {'temperature_bucket'}:
    raise ValueError('Target or weather feature contract changed')

Final sanity checks passed: calendar_weather_candidate 14,640 Nov-Dec rows
Final sanity checks passed: weather_only_candidate 14,640 Nov-Dec rows
